# YOLOv8s: reproduction notebook
This is the standalone Kaggle notebook for the selected SGD experiment.
The notebook is stored without execution outputs; actual results, plots and checkpoint
from the completed run are committed alongside it. See [README_YOLO.md](README_YOLO.md).
Dataset: linxel/dfire-prepared. Test evaluation is disabled by default.


# YOLOv8s: retraining with the baseline time budget
**Fresh run: this notebook does not contain results yet.**

The baseline completed 6 epochs in **14,488.2 seconds = 4 h 1 min 28.2 s**.
This YOLO rerun uses the same maximum training-plus-validation time budget.
Start from the original COCO-pretrained `yolov8s.pt`, not the previous three-hour checkpoint.

The remaining settings are unchanged: SGD, LR 0.005 to 1e-6, momentum 0.9,
weight decay 0.0005, no Nesterov, batch 4, FP32 without AMP or gradient accumulation,
gradient clipping 10, joint horizontal flips with p=0.5, baseline input preprocessing,
seed 42, no warmup and no EMA averaging. Cosine LR decay spans the new time budget.
Select the best checkpoint by validation COCO mAP@0.5:0.95.

Early-stopping patience remains **3**. Training may therefore finish before the time limit.
The maximum available time is matched; actual duration must be checked in the run log.
Use **one Tesla T4 GPU**. Dataset preparation and final evaluation require additional time.
Completing validation may slightly exceed the limit; the final epoch may be incomplete.

The baseline retains cosine decay over 6 epochs, whereas YOLO uses time-based cosine
decay. This comparison limitation remains. A separate output directory and ZIP preserve
the previous results. Final test evaluation is disabled by default.

In [ ]:
%pip install -q "ultralytics==8.3.221" "PyYAML>=6.0.3,<6.1" kagglehub torchmetrics pycocotools

In [ ]:
from pathlib import Path
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/yolo_config'
Path(os.environ['YOLO_CONFIG_DIR']).mkdir(parents=True, exist_ok=True)
OUT = Path('/kaggle/working/yolov8s_baseline_time')
OUT.mkdir(parents=True, exist_ok=True)
ROOT = Path('/kaggle/working/dfire_main_from_prepared')
DATASET_ID = 'linxel/dfire-prepared'
SOURCE_DATA = None
CLASS_NAMES = {0: 'smoke', 1: 'fire'}
EXPECTED_COUNTS = {'train': 14122, 'val': 3099, 'test': 4306}
BATCH = 4
BASELINE_SECONDS = 14488.2
HOURS = BASELINE_SECONDS / 3600.0
RUN_FINAL_TEST = False  # Enable only for the final evaluation of the fixed model and settings.
BASELINE_WEIGHTS = None  # Path to the NEW corrected baseline .pth, if available.
BASELINE_REFERENCE = {'source': 'https://github.com/cvteam19/cv_project/blob/main/baseline/training_history.json', 'reference_history_sha256': 'b69bf3ca8db4cf15130f3f998a96bc6b38c73a3e7958b2a874ba491c08b51c6b', 'baseline_logged_epochs': 6, 'baseline_elapsed_seconds': 14488.2, 'baseline_duration': '4 h 1 min 28.2 s', 'baseline_gpu': 'Tesla T4', 'baseline_scheduler': 'CosineAnnealingLR(T_max=6, eta_min=1e-6)', 'yolo_start': 'fresh COCO-pretrained yolov8s.pt; no resume', 'yolo_time_limit_seconds': 14488.2, 'yolo_patience': 3, 'duration_scope': 'training plus in-loop validation; dataset preparation and final evaluation excluded', 'caveat': 'equal time limits do not guarantee equal elapsed times if early stopping triggers; LR schedules still differ'}

In [ ]:
"""D-Fire main model experiments. Importable helpers; run through the companion notebook."""
from pathlib import Path
import csv
import hashlib
import json
import math
import random
import shutil
import time

NAMES = {0: 'smoke', 1: 'fire'}
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def dump(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')


def write_csv(path, rows, fields=None):
    rows = list(rows)
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def clean_label(text):
    """Match Data Lead policy: reject invalid rows, clip valid boxes to [0, 1]."""
    kept, rejected, clipped = [], [], 0
    for number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        try:
            values = list(map(float, line.split()))
            if len(values) != 5 or not all(map(math.isfinite, values)):
                raise ValueError('expected five finite numbers')
            c, x, y, w, h = values
            if c not in NAMES or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError('invalid class/center/size')
            corners = [x-w/2, y-h/2, x+w/2, y+h/2]
            a, b, d, e = [min(1., max(0., z)) for z in corners]
            if d <= a or e <= b:
                raise ValueError('no visible area')
            clipped += int(corners != [a, b, d, e])
            kept.append(f'{int(c)} {(a+d)/2:.8f} {(b+e)/2:.8f} {d-a:.8f} {e-b:.8f}')
        except ValueError as exc:
            rejected.append({'line': number, 'text': line, 'reason': str(exc)})
    return '\n'.join(kept) + ('\n' if kept else ''), rejected, clipped


def images(root, split):
    return sorted(p for p in (Path(root)/split/'images').rglob('*')
                  if p.is_file() and p.suffix.lower() in EXTENSIONS)


def label_for(root, split, image):
    relative = Path(image).relative_to(Path(root)/split/'images')
    return Path(root)/split/'labels'/relative.with_suffix('.txt')


def locate_prepared(downloaded):
    """Find real image/label folders without trusting stale absolute paths in YAML."""
    downloaded = Path(downloaded)
    candidates = [downloaded] + [p.parent for p in downloaded.rglob('train') if p.is_dir()]
    candidates = sorted(set(p.resolve() for p in candidates if all(
        (p/s/'images').is_dir() and (p/s/'labels').is_dir() for s in ('train', 'val', 'test'))))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one prepared data root, found {candidates}. '
                           'Set SOURCE_DATA to the folder containing train/, val/, test/. '
                           'If the Input contains only an archive, extract it into working first.')
    return candidates[0]


def check_class_mapping(source, expected, search_root=None):
    """Verify published names where available; otherwise record explicit project mapping."""
    source = Path(source).resolve()
    search_root = Path(search_root).resolve() if search_root is not None else source.parent
    configs = sorted(set(search_root.rglob('data*.yaml')) | set(search_root.rglob('data*.yml')))
    verified_files = []
    for path in configs:
        import yaml
        cfg = yaml.safe_load(path.read_text(encoding='utf-8'))
        if not isinstance(cfg, dict) or 'names' not in cfg:
            continue
        names = cfg['names']
        names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
        if names != expected:
            raise ValueError(f'Unexpected class mapping in {path}: {names}; expected {expected}')
        verified_files.append(str(path))
    if not verified_files:
        print(f'No published YAML with names found. Using explicit project mapping: {expected}. '
              'Label IDs will be validated; their semantic names cannot be inferred from numbers alone.')
    return dict(names=expected, verified_from_yaml=bool(verified_files), yaml_files=verified_files,
                mapping_source='published_yaml' if verified_files else 'explicit_project_configuration')


def prepare(source, dest, expected=None, prepared=False):
    """Writable image COPIES avoid Ultralytics repairing JPEGs in read-only inputs."""
    from PIL import Image
    source, dest = Path(source).resolve(), Path(dest).resolve()
    if dest == source or source in dest.parents or dest in source.parents:
        raise ValueError('Source and prepared output must be separate directory trees')
    if dest.exists():
        raise FileExistsError(f'{dest} already exists. Reuse its config or choose a new output.')
    dest.mkdir(parents=True)
    report, rejects, manifest, hashes = {}, [], [], {}
    for split in ('train', 'val', 'test'):
        files = images(source, split)
        if not files or (expected and len(files) != expected[split]):
            raise ValueError(f'{split}: unexpected image count {len(files)}')
        seen_labels, stats = set(), dict(images=len(files), boxes=0, empty_labels=0, clipped=0, rejected=0)
        for src in files:
            label = label_for(source, split, src)
            if not label.is_file():
                raise FileNotFoundError(f'Missing label is not a verified negative: {label}')
            key = label.relative_to(source/split/'labels')
            if key in seen_labels:
                raise ValueError(f'Ambiguous image-to-label mapping: {key}')
            seen_labels.add(key)
            digest = hashlib.sha256(src.read_bytes()).hexdigest()
            if digest in hashes and hashes[digest] != split:
                raise ValueError(f'Exact cross-split duplicate: {src}; other split={hashes[digest]}')
            hashes[digest] = split
            rel = src.relative_to(source/split/'images')
            target = dest/split/'images'/rel
            target.parent.mkdir(parents=True, exist_ok=True)
            # Verify decode without resizing or EXIF transpose (boxes use stored pixel coordinates).
            with Image.open(src) as im:
                im.load()
            shutil.copy2(src, target)
            original_text = label.read_text(encoding='utf-8')
            text, bad, clipped = clean_label(original_text)
            if prepared:
                original_rows = [list(map(float, line.split())) for line in original_text.splitlines() if line.strip()] if not bad else []
                cleaned_rows = [list(map(float, line.split())) for line in text.splitlines()]
                changed = len(original_rows) != len(cleaned_rows) or any(
                    abs(a-b) > 1e-7 for row_a, row_b in zip(original_rows, cleaned_rows)
                    for a, b in zip(row_a, row_b))
                if bad or changed:
                    raise ValueError(f'Prepared label failed validation: {label}. '
                                     'No automatic annotation changes are allowed in prepared mode.')
                text, clipped = original_text, 0
            out_label = label_for(dest, split, target)
            out_label.parent.mkdir(parents=True, exist_ok=True)
            out_label.write_text(text, encoding='utf-8')
            stats['boxes'] += sum(bool(line.strip()) for line in text.splitlines())
            stats['empty_labels'] += int(not text.strip())
            stats['clipped'] += clipped
            stats['rejected'] += len(bad)
            rejects.extend(dict(split=split, image=str(rel), **row) for row in bad)
            manifest.append(dict(split=split, image=str(rel), sha256=digest))
        report[split] = stats
        print(split, stats)
    # Preserve preparation provenance where supplied by the published dataset.
    for directory in (source.parent, source):
        for name in ('removed_label_rows.csv', 'dataset_summary.json', 'data_lead_final_summary.json'):
            report_path = directory/name
            if report_path.is_file():
                (dest/'source_reports').mkdir(exist_ok=True)
                shutil.copy2(report_path, dest/'source_reports'/name)
    write_csv(dest/'rejected_rows.csv', rejects, ['split', 'image', 'line', 'text', 'reason'])
    write_csv(dest/'manifest.csv', manifest)
    dump(dest/'preparation.json', dict(splits=report, prepared_input=prepared,
         source=str(source), labels_modified=not prepared))  # Completion marker written last.
    return report


def make_yaml(path, root, train=None):
    import yaml
    config = dict(path=str(Path(root).resolve()), train=train or 'train/images',
                  val='val/images', test='test/images', names=NAMES, nc=2)
    Path(path).write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    return str(Path(path).resolve())


def load_model(weights, architecture='yolo'):
    from ultralytics import YOLO, RTDETR
    return (RTDETR if architecture == 'rtdetr' else YOLO)(str(weights))


def train(weights, data, out, name, epochs=50, batch=8, lr=0.001,
          architecture='yolo', patience=8, resume=None):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
    run = Path(out)/name
    if resume:
        model = load_model(resume, architecture)
        model.train(resume=True, device=0)
    else:
        if run.exists():
            raise FileExistsError(f'{run} exists. Use resume=last.pt or a new name.')
        model = load_model(weights, architecture)
        start = time.time()
        model.train(data=str(data), epochs=epochs, imgsz=640, batch=batch,
                    device=0, workers=2, optimizer='AdamW', lr0=lr, lrf=0.01,
                    weight_decay=0.0005, warmup_epochs=3, cos_lr=True,
                    seed=42, deterministic=True, amp=True, cache=False,
                    patience=patience, close_mosaic=min(5, epochs),
                    mosaic=1.0, mixup=0.0, fliplr=0.5, flipud=0.0,
                    save=True, save_period=5, plots=True,
                    project=str(out), name=name, exist_ok=False)
        dump(run/'elapsed.json', {'train_seconds': time.time()-start})
    save_dir = Path(model.trainer.save_dir)
    best = save_dir/'weights'/'best.pt'
    if not best.is_file():
        raise FileNotFoundError(best)
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return str(best)


def evaluate(weights, data, out, name, size=640, split='val', architecture='yolo'):
    model = load_model(weights, architecture)
    metrics = model.val(data=str(data), split=split, imgsz=size, batch=4,
                        device=0, workers=2, conf=0.001, iou=0.7, max_det=300,
                        rect=False, plots=True, project=str(out), name=name,
                        exist_ok=False)
    row = dict(name=name, checkpoint=str(weights), architecture=architecture,
               split=split, imgsz=size, map50=float(metrics.box.map50),
               map50_95=float(metrics.box.map),
               inference_ms_per_image=float(metrics.speed['inference']))
    # AP class indices are explicit; absent classes must not inherit the mean AP.
    for cls, ap in zip(metrics.box.ap_class_index, metrics.box.ap):
        row[f'{NAMES[int(cls)]}_map50_95'] = float(ap)
    dump(Path(metrics.save_dir)/'summary.json', row)
    return row


def ground_truth(root, split, image):
    result = []
    for line in label_for(root, split, image).read_text().splitlines():
        c, x, y, w, h = map(float, line.split())
        result.append([int(c), x-w/2, y-h/2, x+w/2, y+h/2])
    return result


def predict_records(weights, root, split, size=640, architecture='yolo', subset=None):
    model = load_model(weights, architecture)
    paths = subset if subset is not None else images(root, split)
    records = []
    # Bound the source list per call; stream results, retaining only boxes, not image arrays.
    for start in range(0, len(paths), 128):
        results = model.predict(source=[str(p) for p in paths[start:start+128]],
                                stream=True, imgsz=size, batch=4, device=0,
                                conf=0.001, iou=0.7, max_det=300, rect=False, verbose=False)
        for result in results:
            boxes = result.boxes
            predictions = [[int(c), float(s), *map(float, b)] for c, s, b in
                           zip(boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist(),
                               boxes.xyxyn.cpu().tolist())]
            records.append(dict(image=str(result.path),
                                gt=ground_truth(root, split, result.path), pred=predictions))
    if len(records) != len(paths):
        raise RuntimeError('Prediction count mismatch')
    return records


def iou(a, b):
    intersection = max(0., min(a[2], b[2])-max(a[0], b[0])) * max(0., min(a[3], b[3])-max(a[1], b[1]))
    area = lambda box: max(0., box[2]-box[0])*max(0., box[3]-box[1])
    union = area(a)+area(b)-intersection
    return intersection/union if union else 0.


def operating_metrics(records, threshold, cls=None):
    tp = fp = fn = negative_count = false_alarms = 0
    for record in records:
        gt = [g for g in record['gt'] if cls is None or g[0] == cls]
        predictions = sorted([p for p in record['pred'] if p[1] >= threshold and
                              (cls is None or p[0] == cls)], key=lambda p: -p[1])
        used = set()
        for p in predictions:
            candidates = [(iou(p[2:], g[1:]), j) for j, g in enumerate(gt)
                          if j not in used and g[0] == p[0]]
            overlap, j = max(candidates, default=(0., -1))
            if overlap >= 0.5:
                tp += 1
                used.add(j)
            else:
                fp += 1
        fn += len(gt)-len(used)
        # Negative means no annotations of either class, including for per-class FAR.
        if not record['gt']:
            negative_count += 1
            false_alarms += int(bool(predictions))
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    return dict(threshold=threshold, class_name=NAMES.get(cls, 'all'), tp=tp, fp=fp, fn=fn,
                precision=precision, recall=recall,
                f1=2*precision*recall/(precision+recall) if precision+recall else 0.,
                negative_images=negative_count, false_alarm_images=false_alarms,
                false_alarm_rate=false_alarms/negative_count if negative_count else None)


def mine(weights, root, out, limit=200):
    """Rank prepared empty-label train images; exclude known rejected-label cases from audit reports."""
    source = Path(root)
    negatives = [p for p in images(source, 'train') if not label_for(source, 'train', p).read_text().strip()]
    # preparation's rejected_rows prevents mislabeled positives from becoming hard negatives.
    rejected = set()
    for report_path in (source/'rejected_rows.csv', source/'source_reports'/'removed_label_rows.csv'):
        if report_path.is_file():
            with report_path.open(encoding='utf-8') as f:
                rejected.update(r['image'].replace('\\', '/') for r in csv.DictReader(f)
                                if r.get('split') == 'train' and r.get('image'))
    negatives = [p for p in negatives if p.relative_to(source/'train'/'images').as_posix() not in rejected]
    records = predict_records(weights, root, 'train', subset=negatives)
    ranked = sorted(records, key=lambda r: max([p[1] for p in r['pred']], default=0.), reverse=True)
    rows = [dict(image=r['image'], max_conf=max(p[1] for p in r['pred']),
                 approved='', category='', notes='') for r in ranked if r['pred']][:limit]
    write_csv(Path(out)/'hard_negative_review.csv', rows,
              ['image', 'max_conf', 'approved', 'category', 'notes'])
    # Paginated contact sheets; inspect full images separately when uncertain.
    import matplotlib.pyplot as plt
    from PIL import Image
    for start in range(0, len(rows), 20):
        fig, axes = plt.subplots(4, 5, figsize=(18, 12))
        for ax, row in zip(axes.flat, rows[start:start+20]):
            with Image.open(row['image']) as im:
                ax.imshow(im.convert('RGB'))
            ax.set_title(f"{Path(row['image']).name}\nscore={row['max_conf']:.3f}", fontsize=8)
        for ax in axes.flat:
            ax.axis('off')
        fig.tight_layout()
        fig.savefig(Path(out)/f'hard_negative_review_{start//20:02d}.jpg', dpi=100)
        plt.close(fig)
    return rows


def ablation_lists(root, review, out, repeats=2):
    """Equal length = equal optimizer steps; base data retained in both arms."""
    base = [str(p.resolve()) for p in images(root, 'train')]
    membership = set(base)
    with Path(review).open(encoding='utf-8') as f:
        reviewed = list(csv.DictReader(f))
    approved = sorted({str(Path(r['image']).resolve()) for r in reviewed
                       if r['approved'].strip().lower() == 'yes'})
    if not approved:
        raise ValueError('Review candidates first; set approved=yes only for confirmed backgrounds.')
    for p in approved:
        if p not in membership or label_for(root, 'train', p).read_text().strip():
            raise ValueError(f'Approved image is not an empty-label TRAIN image: {p}')
    negatives = [p for p in base if not label_for(root, 'train', p).read_text().strip()]
    count = len(approved)*repeats
    rng = random.Random(42)
    # Controls repeat randomly sampled negatives; treatment repeats reviewed hard negatives.
    control = base + rng.choices(negatives, k=count)
    treatment = base + approved*repeats
    assert len(control) == len(treatment)
    paths = []
    for name, entries in [('control', control), ('hard_negative', treatment)]:
        listing = Path(out)/f'{name}_train.txt'
        listing.write_text('\n'.join(entries)+'\n', encoding='utf-8')
        paths.append(make_yaml(Path(out)/f'{name}.yaml', root, str(listing.resolve())))
    dump(Path(out)/'ablation_design.json', dict(base_images=len(base), extra_samples=count,
         images_per_epoch=len(control), approved=approved, seed=42,
         policy='same checkpoint, epochs, optimizer, batch; random negatives vs reviewed hard negatives'))
    return paths

In [ ]:
import sys, subprocess, platform
import torch, ultralytics, yaml
import kagglehub
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle: Settings > Accelerator.'
assert 'T4' in torch.cuda.get_device_name(0), 'Select a Tesla T4, matching the baseline GPU for the time-budget comparison.'
environment = dict(python=sys.version, platform=platform.platform(), torch=torch.__version__,
                   ultralytics=ultralytics.__version__, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0))
dump(OUT/'environment.json', environment)
(OUT/'pip_freeze.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print(environment)

if not ROOT.exists():
    downloaded = None
    if SOURCE_DATA is None:
        downloaded = Path(kagglehub.dataset_download(DATASET_ID))
        SOURCE_DATA = locate_prepared(downloaded)
    SOURCE_DATA = Path(SOURCE_DATA).resolve()
    assert CLASS_NAMES == NAMES, 'Class mapping must match the project: 0=smoke, 1=fire.'
    mapping = check_class_mapping(SOURCE_DATA, CLASS_NAMES, search_root=downloaded)
    prepare(SOURCE_DATA, ROOT, EXPECTED_COUNTS, prepared=True)
    dump(ROOT/'dataset_source.json', dict(dataset_id=DATASET_ID, source=str(SOURCE_DATA),
                                         prepared_input=True, labels_modified=False, class_mapping=mapping))
else:
    assert (ROOT/'preparation.json').is_file(), 'Dataset preparation was interrupted. Set a new ROOT and retry.'
    assert json.loads((ROOT/'preparation.json').read_text())['prepared_input'], 'ROOT must contain data copied from the prepared dataset.'
    for split, count in EXPECTED_COUNTS.items():
        assert len(images(ROOT, split)) == count
DATA_YAML = make_yaml(OUT/'data.yaml', ROOT)
dump(OUT/'dataset_source.json', json.loads((ROOT/'dataset_source.json').read_text()))
print('Data ready:', DATA_YAML)

## Verify input preprocessing and SGD before training
The numerical checks compare input tensors and bounding boxes with the actual
Faster R-CNN transform, and verify joint flipping and SGD updates. They must pass
on Kaggle before training. Checkpoint selection uses torchmetrics COCO mAP, the same
implementation used in the final evaluation. Native Ultralytics plots supplement
the exported metric tables.

In [ ]:
"""Shared LR protocol for both detectors: SGD + a three-hour cosine budget."""
import math


def budget_cosine_lr(elapsed_seconds, budget_seconds=10800.0,
                     initial_lr=0.005, minimum_lr=1e-6):
    if budget_seconds <= 0:
        raise ValueError('budget_seconds must be positive')
    progress = min(1.0, max(0.0, float(elapsed_seconds) / budget_seconds))
    return minimum_lr + (initial_lr - minimum_lr) * (1 + math.cos(math.pi * progress)) / 2


def set_budget_lr(optimizer, elapsed_seconds, budget_seconds=10800.0):
    lr = budget_cosine_lr(elapsed_seconds, budget_seconds)
    for group in optimizer.param_groups:
        group['lr'] = lr
    return lr


def build_shared_sgd(model):
    import torch
    return torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                           lr=0.005, momentum=0.9, weight_decay=0.0005,
                           nesterov=False)


"""Final agreed YOLO: Faster R-CNN input transform, SGD, three-hour budget.

Pinned to ultralytics 8.3.221. Embedded into the standalone Kaggle notebook.
The runtime checks must pass on Kaggle before training.
"""
from copy import copy
from pathlib import Path
import random
import time

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from ultralytics.data import build_dataloader
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.utils import ops

BASELINE_MEAN = [0.485, 0.456, 0.406]
BASELINE_STD = [0.229, 0.224, 0.225]


def input_transform():
    # Same normalization, interpolation, resize rounding, zero padding and stride
    # as Faster R-CNN's native GeneralizedRCNNTransform.
    return GeneralizedRCNNTransform(
        min_size=640, max_size=640, image_mean=BASELINE_MEAN,
        image_std=BASELINE_STD, size_divisible=32)


def flip_image_target(image, boxes):
    reflected = boxes.clone()
    reflected[:, 0] = image.shape[-1] - boxes[:, 2]
    reflected[:, 2] = image.shape[-1] - boxes[:, 0]
    return image.flip(-1), reflected


def collate_baseline_input(samples):
    tensors, targets, filenames, shapes = zip(*samples)
    transform = input_transform()
    # min_size has one value, so train/eval select exactly the same size.
    transformed, targets = transform(list(tensors), list(targets))
    height, width = transformed.tensors.shape[-2:]
    boxes, classes, indices = [], [], []
    for index, target in enumerate(targets):
        xywh = ops.xyxy2xywh(target['boxes'])
        xywh /= xywh.new_tensor([width, height, width, height])
        boxes.append(xywh)
        classes.append(target['labels'].float().reshape(-1, 1))
        indices.append(torch.full((len(xywh),), index, dtype=torch.float32))
    return dict(
        img=transformed.tensors, bboxes=torch.cat(boxes),
        cls=torch.cat(classes), batch_idx=torch.cat(indices),
        im_file=filenames, ori_shape=shapes,
        resized_shape=transformed.image_sizes,
        ratio_pad=tuple(((h1 / h0, w1 / w0), (0, 0)) for
                        (h0, w0), (h1, w1) in zip(shapes, transformed.image_sizes)),
    )


class BaselineInputDataset(Dataset):
    rect = False
    collate_fn = staticmethod(collate_baseline_input)

    def __init__(self, image_dir, train=False):
        self.train = train
        image_dir = Path(image_dir)
        assert image_dir.is_dir(), f'Expected an images directory: {image_dir}'
        self.im_files = sorted(str(p) for p in image_dir.rglob('*')
                               if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
        self.labels = []
        for filename in self.im_files:
            relative = Path(filename).relative_to(image_dir).with_suffix('.txt')
            label = image_dir.parent / 'labels' / relative
            rows = np.array([list(map(float, line.split())) for line in
                             label.read_text().splitlines() if line.strip()], dtype=np.float32).reshape(-1, 5)
            self.labels.append(dict(cls=rows[:, :1], bboxes=rows[:, 1:]))
        assert self.im_files, 'No images found'

    def __len__(self):
        return len(self.im_files)

    def __getitem__(self, index):
        filename = self.im_files[index]
        with Image.open(filename) as image:
            image = to_tensor(image.convert('RGB'))
        h, w = image.shape[-2:]
        annotation = self.labels[index]
        boxes = ops.xywh2xyxy(torch.from_numpy(annotation['bboxes'].copy()))
        boxes *= boxes.new_tensor([w, h, w, h])
        if self.train and random.random() < 0.5:
            image, boxes = flip_image_target(image, boxes)
        return image, dict(boxes=boxes, labels=torch.from_numpy(
            annotation['cls'].copy()).reshape(-1).long()), filename, (h, w)


def move_baseline_batch(batch, device):
    # Already float32 and normalized exactly once. No YOLO /255 or letterbox.
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            batch[key] = value.to(device, non_blocking=device.type == 'cuda')
    assert batch['img'].dtype == torch.float32
    return batch


def display_batch(batch):
    # Plotting only: invert ImageNet normalization, never change training inputs.
    result = dict(batch)
    image = batch['img']
    result['img'] = (image * image.new_tensor(BASELINE_STD)[None, :, None, None]
                     + image.new_tensor(BASELINE_MEAN)[None, :, None, None]).clamp(0, 1)
    return result


class BaselineInputValidator(DetectionValidator):
    def init_metrics(self, model):
        super().init_metrics(model)
        from torchmetrics.detection.mean_ap import MeanAveragePrecision
        self.common_map = MeanAveragePrecision(box_format='xyxy', iou_type='bbox',
            max_detection_thresholds=[1, 10, 100])

    def update_metrics(self, preds, batch):
        super().update_metrics(preds, batch)
        for index, pred in enumerate(preds):
            truth = self._prepare_batch(index, batch)
            scaled = self.scale_preds(pred, truth)
            gt = self.scale_preds(dict(bboxes=truth['bboxes']), truth)
            self.common_map.update([dict(boxes=scaled['bboxes'].detach().cpu(),
                labels=scaled['cls'].long().detach().cpu(), scores=scaled['conf'].detach().cpu())],
                [dict(boxes=gt['bboxes'].detach().cpu(), labels=truth['cls'].long().detach().cpu())])

    def get_stats(self):
        stats = super().get_stats()
        common = self.common_map.compute()
        stats['metrics/mAP50(B)'] = float(common['map_50'])
        stats['metrics/mAP50-95(B)'] = float(common['map'])
        stats['fitness'] = float(common['map'])
        return stats

    def preprocess(self, batch):
        assert not self.args.half, 'Use FP32 for this protocol'
        return move_baseline_batch(batch, self.device)

    def build_dataset(self, img_path, mode='val', batch=None):
        return BaselineInputDataset(img_path, train=False)

    def scale_preds(self, pred, pbatch):
        # Exact per-axis inverse of torchvision's resize, no centered padding.
        result = {key: value.clone() for key, value in pred.items()}
        sy, sx = pbatch['ratio_pad'][0]
        result['bboxes'][:, [0, 2]] /= sx
        result['bboxes'][:, [1, 3]] /= sy
        ops.clip_boxes(result['bboxes'], pbatch['ori_shape'])
        return result

    def plot_val_samples(self, batch, ni):
        return super().plot_val_samples(display_batch(batch), ni)

    def plot_predictions(self, batch, preds, ni):
        return super().plot_predictions(display_batch(batch), preds, ni)


class AgreedYoloTrainer(DetectionTrainer):
    def build_dataset(self, img_path, mode='train', batch=None):
        return BaselineInputDataset(img_path, train=mode == 'train')

    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode='train'):
        dataset = self.build_dataset(dataset_path, mode)
        return build_dataloader(dataset, batch=int(self.args.batch),
                                workers=self.args.workers, shuffle=mode == 'train', rank=rank)

    def preprocess_batch(self, batch):
        return move_baseline_batch(batch, self.device)

    def build_optimizer(self, model, name='SGD', lr=0.005, momentum=0.9,
                        decay=0.0005, iterations=1e5):
        assert name == 'SGD'
        # Match baseline: one group, decay on every trainable parameter, no Nesterov.
        return torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
            lr=lr, momentum=momentum, weight_decay=decay, nesterov=False)

    def _setup_scheduler(self):
        # Native trainer expects an epoch scheduler, but the actual LR is set
        # below before EVERY optimizer update, using the shared wall-time rule.
        self.lf = lambda epoch: 1.0
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lambda epoch: 1.0)

    def optimizer_step(self):
        set_budget_lr(self.optimizer, time.time() - self.train_time_start,
                      self.args.time * 3600)
        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad(set_to_none=True)
        # Baseline has no EMA. Retain Ultralytics' checkpoint container, but copy
        # current weights instead of averaging them. Validator uses these weights.
        if self.ema:
            self.ema.ema.load_state_dict(self.model.state_dict())
            self.ema.updates += 1

    def get_validator(self):
        self.loss_names = ('box_loss', 'cls_loss', 'dfl_loss')
        return BaselineInputValidator(self.test_loader, save_dir=self.save_dir,
                                      args=copy(self.args), _callbacks=self.callbacks)

    def plot_training_samples(self, batch, ni):
        return super().plot_training_samples(display_batch(batch), ni)


@torch.inference_mode()
def predict_with_baseline_input(core_model, pil_images, device='cuda',
                                score_floor=0.001, nms_iou=0.7, max_det=100):
    """Inference adapter for YOLO(...).model, returning original-pixel boxes.

    Use instead of YOLO.predict() for these newly trained checkpoints.
    Includes tensor conversion, shared preprocessing and model postprocessing.
    """
    from ultralytics.utils import nms
    tensors = [to_tensor(image.convert('RGB')) for image in pil_images]
    original_shapes = [tuple(image.shape[-2:]) for image in tensors]
    transformed, _ = input_transform()(tensors, None)
    inputs = transformed.tensors.to(device)
    raw = core_model(inputs)
    outputs = nms.non_max_suppression(raw, conf_thres=score_floor,
        iou_thres=nms_iou, nc=2, multi_label=True, max_det=max_det)
    result = []
    for output, (h0, w0), (h1, w1) in zip(outputs, original_shapes, transformed.image_sizes):
        boxes = output[:, :4].clone()
        boxes[:, [0, 2]] *= w0 / w1
        boxes[:, [1, 3]] *= h0 / h1
        ops.clip_boxes(boxes, (h0, w0))
        result.append(dict(boxes=boxes, scores=output[:, 4], labels=output[:, 5].long()))
    return result


def check_baseline_equivalence():
    # Deliberately include rectangular inputs and an empty target.
    generator = torch.Generator().manual_seed(42)
    tensors = [torch.rand((3, 173, 311), generator=generator),
               torch.rand((3, 487, 199), generator=generator)]
    targets = [dict(boxes=torch.tensor([[11., 7., 89., 142.]]), labels=torch.tensor([0])),
               dict(boxes=torch.empty((0, 4)), labels=torch.empty((0,), dtype=torch.long))]
    samples = [(image, target, f'synthetic-{i}', tuple(image.shape[-2:]))
               for i, (image, target) in enumerate(zip(tensors, targets))]
    actual = collate_baseline_input(samples)
    # An actual Faster R-CNN model's transform, with its head/network not needed.
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    reference_model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                            min_size=640, max_size=640)
    reference, transformed_targets = reference_model.transform(tensors, targets)
    torch.testing.assert_close(actual['img'], reference.tensors, rtol=0, atol=0)
    recovered = ops.xywh2xyxy(actual['bboxes']) * actual['bboxes'].new_tensor(
        [actual['img'].shape[-1], actual['img'].shape[-2]] * 2)
    torch.testing.assert_close(recovered, transformed_targets[0]['boxes'], atol=1e-4, rtol=1e-6)
    reflected, boxes = flip_image_target(tensors[0], targets[0]['boxes'])
    torch.testing.assert_close(reflected, tensors[0].flip(-1), rtol=0, atol=0)
    torch.testing.assert_close(boxes, torch.tensor([[222., 7., 300., 142.]]), rtol=0, atol=0)
    restored_image, restored_boxes = flip_image_target(reflected, boxes)
    torch.testing.assert_close(restored_image, tensors[0], rtol=0, atol=0)
    torch.testing.assert_close(restored_boxes, targets[0]['boxes'], rtol=0, atol=0)
    # Optimizer updates match baseline SGD. Time schedule is the new shared rule.
    parameter = torch.nn.Parameter(torch.ones(2))
    reference_parameter = torch.nn.Parameter(torch.ones(2))
    adapter = object.__new__(AgreedYoloTrainer)
    tiny = torch.nn.ParameterList([parameter])
    adapter.optimizer = adapter.build_optimizer(tiny)
    reference_optimizer = torch.optim.SGD([reference_parameter], lr=0.005,
        momentum=0.9, weight_decay=0.0005, nesterov=False)
    for elapsed in [0, 3600, 5400, 9000, 10800]:
        lr = budget_cosine_lr(elapsed, initial_lr=0.005, minimum_lr=1e-6)
        for group in adapter.optimizer.param_groups:
            group['lr'] = lr
        for group in reference_optimizer.param_groups:
            group['lr'] = lr
        assert adapter.optimizer.param_groups[0]['lr'] == reference_optimizer.param_groups[0]['lr']
        parameter.grad = torch.ones(2)
        reference_parameter.grad = torch.ones(2)
        adapter.optimizer.step()
        reference_optimizer.step()
        torch.testing.assert_close(parameter, reference_parameter, rtol=0, atol=0)
        assert lr >= 1e-6
    print('Input tensors, boxes, flip, SGD updates and budget schedule: OK')



check_baseline_equivalence()
(OUT/'budget_protocol.py').write_text('"""Shared LR protocol for both detectors: SGD + a three-hour cosine budget."""\nimport math\n\n\ndef budget_cosine_lr(elapsed_seconds, budget_seconds=10800.0,\n                     initial_lr=0.005, minimum_lr=1e-6):\n    if budget_seconds <= 0:\n        raise ValueError(\'budget_seconds must be positive\')\n    progress = min(1.0, max(0.0, float(elapsed_seconds) / budget_seconds))\n    return minimum_lr + (initial_lr - minimum_lr) * (1 + math.cos(math.pi * progress)) / 2\n\n\ndef set_budget_lr(optimizer, elapsed_seconds, budget_seconds=10800.0):\n    lr = budget_cosine_lr(elapsed_seconds, budget_seconds)\n    for group in optimizer.param_groups:\n        group[\'lr\'] = lr\n    return lr\n\n\ndef build_shared_sgd(model):\n    import torch\n    return torch.optim.SGD([p for p in model.parameters() if p.requires_grad],\n                           lr=0.005, momentum=0.9, weight_decay=0.0005,\n                           nesterov=False)\n', encoding='utf-8')
(OUT/'final_yolo_protocol.py').write_text('"""Final agreed YOLO: Faster R-CNN input transform, SGD, three-hour budget.\n\nPinned to ultralytics 8.3.221. Embedded into the standalone Kaggle notebook.\nThe runtime checks must pass on Kaggle before training.\n"""\nfrom copy import copy\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms.functional import to_tensor\nfrom torchvision.models.detection.transform import GeneralizedRCNNTransform\nfrom ultralytics.data import build_dataloader\nfrom ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator\nfrom ultralytics.utils import ops\nfrom budget_protocol import budget_cosine_lr, set_budget_lr\n\nBASELINE_MEAN = [0.485, 0.456, 0.406]\nBASELINE_STD = [0.229, 0.224, 0.225]\n\n\ndef input_transform():\n    # Same normalization, interpolation, resize rounding, zero padding and stride\n    # as Faster R-CNN\'s native GeneralizedRCNNTransform.\n    return GeneralizedRCNNTransform(\n        min_size=640, max_size=640, image_mean=BASELINE_MEAN,\n        image_std=BASELINE_STD, size_divisible=32)\n\n\ndef flip_image_target(image, boxes):\n    reflected = boxes.clone()\n    reflected[:, 0] = image.shape[-1] - boxes[:, 2]\n    reflected[:, 2] = image.shape[-1] - boxes[:, 0]\n    return image.flip(-1), reflected\n\n\ndef collate_baseline_input(samples):\n    tensors, targets, filenames, shapes = zip(*samples)\n    transform = input_transform()\n    # min_size has one value, so train/eval select exactly the same size.\n    transformed, targets = transform(list(tensors), list(targets))\n    height, width = transformed.tensors.shape[-2:]\n    boxes, classes, indices = [], [], []\n    for index, target in enumerate(targets):\n        xywh = ops.xyxy2xywh(target[\'boxes\'])\n        xywh /= xywh.new_tensor([width, height, width, height])\n        boxes.append(xywh)\n        classes.append(target[\'labels\'].float().reshape(-1, 1))\n        indices.append(torch.full((len(xywh),), index, dtype=torch.float32))\n    return dict(\n        img=transformed.tensors, bboxes=torch.cat(boxes),\n        cls=torch.cat(classes), batch_idx=torch.cat(indices),\n        im_file=filenames, ori_shape=shapes,\n        resized_shape=transformed.image_sizes,\n        ratio_pad=tuple(((h1 / h0, w1 / w0), (0, 0)) for\n                        (h0, w0), (h1, w1) in zip(shapes, transformed.image_sizes)),\n    )\n\n\nclass BaselineInputDataset(Dataset):\n    rect = False\n    collate_fn = staticmethod(collate_baseline_input)\n\n    def __init__(self, image_dir, train=False):\n        self.train = train\n        image_dir = Path(image_dir)\n        assert image_dir.is_dir(), f\'Expected an images directory: {image_dir}\'\n        self.im_files = sorted(str(p) for p in image_dir.rglob(\'*\')\n                               if p.suffix.lower() in {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'})\n        self.labels = []\n        for filename in self.im_files:\n            relative = Path(filename).relative_to(image_dir).with_suffix(\'.txt\')\n            label = image_dir.parent / \'labels\' / relative\n            rows = np.array([list(map(float, line.split())) for line in\n                             label.read_text().splitlines() if line.strip()], dtype=np.float32).reshape(-1, 5)\n            self.labels.append(dict(cls=rows[:, :1], bboxes=rows[:, 1:]))\n        assert self.im_files, \'No images found\'\n\n    def __len__(self):\n        return len(self.im_files)\n\n    def __getitem__(self, index):\n        filename = self.im_files[index]\n        with Image.open(filename) as image:\n            image = to_tensor(image.convert(\'RGB\'))\n        h, w = image.shape[-2:]\n        annotation = self.labels[index]\n        boxes = ops.xywh2xyxy(torch.from_numpy(annotation[\'bboxes\'].copy()))\n        boxes *= boxes.new_tensor([w, h, w, h])\n        if self.train and random.random() < 0.5:\n            image, boxes = flip_image_target(image, boxes)\n        return image, dict(boxes=boxes, labels=torch.from_numpy(\n            annotation[\'cls\'].copy()).reshape(-1).long()), filename, (h, w)\n\n\ndef move_baseline_batch(batch, device):\n    # Already float32 and normalized exactly once. No YOLO /255 or letterbox.\n    for key, value in batch.items():\n        if isinstance(value, torch.Tensor):\n            batch[key] = value.to(device, non_blocking=device.type == \'cuda\')\n    assert batch[\'img\'].dtype == torch.float32\n    return batch\n\n\ndef display_batch(batch):\n    # Plotting only: invert ImageNet normalization, never change training inputs.\n    result = dict(batch)\n    image = batch[\'img\']\n    result[\'img\'] = (image * image.new_tensor(BASELINE_STD)[None, :, None, None]\n                     + image.new_tensor(BASELINE_MEAN)[None, :, None, None]).clamp(0, 1)\n    return result\n\n\nclass BaselineInputValidator(DetectionValidator):\n    def init_metrics(self, model):\n        super().init_metrics(model)\n        from torchmetrics.detection.mean_ap import MeanAveragePrecision\n        self.common_map = MeanAveragePrecision(box_format=\'xyxy\', iou_type=\'bbox\',\n            max_detection_thresholds=[1, 10, 100])\n\n    def update_metrics(self, preds, batch):\n        super().update_metrics(preds, batch)\n        for index, pred in enumerate(preds):\n            truth = self._prepare_batch(index, batch)\n            scaled = self.scale_preds(pred, truth)\n            gt = self.scale_preds(dict(bboxes=truth[\'bboxes\']), truth)\n            self.common_map.update([dict(boxes=scaled[\'bboxes\'].detach().cpu(),\n                labels=scaled[\'cls\'].long().detach().cpu(), scores=scaled[\'conf\'].detach().cpu())],\n                [dict(boxes=gt[\'bboxes\'].detach().cpu(), labels=truth[\'cls\'].long().detach().cpu())])\n\n    def get_stats(self):\n        stats = super().get_stats()\n        common = self.common_map.compute()\n        stats[\'metrics/mAP50(B)\'] = float(common[\'map_50\'])\n        stats[\'metrics/mAP50-95(B)\'] = float(common[\'map\'])\n        stats[\'fitness\'] = float(common[\'map\'])\n        return stats\n\n    def preprocess(self, batch):\n        assert not self.args.half, \'Use FP32 for this protocol\'\n        return move_baseline_batch(batch, self.device)\n\n    def build_dataset(self, img_path, mode=\'val\', batch=None):\n        return BaselineInputDataset(img_path, train=False)\n\n    def scale_preds(self, pred, pbatch):\n        # Exact per-axis inverse of torchvision\'s resize, no centered padding.\n        result = {key: value.clone() for key, value in pred.items()}\n        sy, sx = pbatch[\'ratio_pad\'][0]\n        result[\'bboxes\'][:, [0, 2]] /= sx\n        result[\'bboxes\'][:, [1, 3]] /= sy\n        ops.clip_boxes(result[\'bboxes\'], pbatch[\'ori_shape\'])\n        return result\n\n    def plot_val_samples(self, batch, ni):\n        return super().plot_val_samples(display_batch(batch), ni)\n\n    def plot_predictions(self, batch, preds, ni):\n        return super().plot_predictions(display_batch(batch), preds, ni)\n\n\nclass AgreedYoloTrainer(DetectionTrainer):\n    def build_dataset(self, img_path, mode=\'train\', batch=None):\n        return BaselineInputDataset(img_path, train=mode == \'train\')\n\n    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode=\'train\'):\n        dataset = self.build_dataset(dataset_path, mode)\n        return build_dataloader(dataset, batch=int(self.args.batch),\n                                workers=self.args.workers, shuffle=mode == \'train\', rank=rank)\n\n    def preprocess_batch(self, batch):\n        return move_baseline_batch(batch, self.device)\n\n    def build_optimizer(self, model, name=\'SGD\', lr=0.005, momentum=0.9,\n                        decay=0.0005, iterations=1e5):\n        assert name == \'SGD\'\n        # Match baseline: one group, decay on every trainable parameter, no Nesterov.\n        return torch.optim.SGD([p for p in model.parameters() if p.requires_grad],\n            lr=lr, momentum=momentum, weight_decay=decay, nesterov=False)\n\n    def _setup_scheduler(self):\n        # Native trainer expects an epoch scheduler, but the actual LR is set\n        # below before EVERY optimizer update, using the shared wall-time rule.\n        self.lf = lambda epoch: 1.0\n        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lambda epoch: 1.0)\n\n    def optimizer_step(self):\n        set_budget_lr(self.optimizer, time.time() - self.train_time_start,\n                      self.args.time * 3600)\n        self.scaler.unscale_(self.optimizer)\n        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)\n        self.scaler.step(self.optimizer)\n        self.scaler.update()\n        self.optimizer.zero_grad(set_to_none=True)\n        # Baseline has no EMA. Retain Ultralytics\' checkpoint container, but copy\n        # current weights instead of averaging them. Validator uses these weights.\n        if self.ema:\n            self.ema.ema.load_state_dict(self.model.state_dict())\n            self.ema.updates += 1\n\n    def get_validator(self):\n        self.loss_names = (\'box_loss\', \'cls_loss\', \'dfl_loss\')\n        return BaselineInputValidator(self.test_loader, save_dir=self.save_dir,\n                                      args=copy(self.args), _callbacks=self.callbacks)\n\n    def plot_training_samples(self, batch, ni):\n        return super().plot_training_samples(display_batch(batch), ni)\n\n\n@torch.inference_mode()\ndef predict_with_baseline_input(core_model, pil_images, device=\'cuda\',\n                                score_floor=0.001, nms_iou=0.7, max_det=100):\n    """Inference adapter for YOLO(...).model, returning original-pixel boxes.\n\n    Use instead of YOLO.predict() for these newly trained checkpoints.\n    Includes tensor conversion, shared preprocessing and model postprocessing.\n    """\n    from ultralytics.utils import nms\n    tensors = [to_tensor(image.convert(\'RGB\')) for image in pil_images]\n    original_shapes = [tuple(image.shape[-2:]) for image in tensors]\n    transformed, _ = input_transform()(tensors, None)\n    inputs = transformed.tensors.to(device)\n    raw = core_model(inputs)\n    outputs = nms.non_max_suppression(raw, conf_thres=score_floor,\n        iou_thres=nms_iou, nc=2, multi_label=True, max_det=max_det)\n    result = []\n    for output, (h0, w0), (h1, w1) in zip(outputs, original_shapes, transformed.image_sizes):\n        boxes = output[:, :4].clone()\n        boxes[:, [0, 2]] *= w0 / w1\n        boxes[:, [1, 3]] *= h0 / h1\n        ops.clip_boxes(boxes, (h0, w0))\n        result.append(dict(boxes=boxes, scores=output[:, 4], labels=output[:, 5].long()))\n    return result\n\n\ndef check_baseline_equivalence():\n    # Deliberately include rectangular inputs and an empty target.\n    generator = torch.Generator().manual_seed(42)\n    tensors = [torch.rand((3, 173, 311), generator=generator),\n               torch.rand((3, 487, 199), generator=generator)]\n    targets = [dict(boxes=torch.tensor([[11., 7., 89., 142.]]), labels=torch.tensor([0])),\n               dict(boxes=torch.empty((0, 4)), labels=torch.empty((0,), dtype=torch.long))]\n    samples = [(image, target, f\'synthetic-{i}\', tuple(image.shape[-2:]))\n               for i, (image, target) in enumerate(zip(tensors, targets))]\n    actual = collate_baseline_input(samples)\n    # An actual Faster R-CNN model\'s transform, with its head/network not needed.\n    from torchvision.models.detection import fasterrcnn_resnet50_fpn\n    reference_model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,\n                                            min_size=640, max_size=640)\n    reference, transformed_targets = reference_model.transform(tensors, targets)\n    torch.testing.assert_close(actual[\'img\'], reference.tensors, rtol=0, atol=0)\n    recovered = ops.xywh2xyxy(actual[\'bboxes\']) * actual[\'bboxes\'].new_tensor(\n        [actual[\'img\'].shape[-1], actual[\'img\'].shape[-2]] * 2)\n    torch.testing.assert_close(recovered, transformed_targets[0][\'boxes\'], atol=1e-4, rtol=1e-6)\n    reflected, boxes = flip_image_target(tensors[0], targets[0][\'boxes\'])\n    torch.testing.assert_close(reflected, tensors[0].flip(-1), rtol=0, atol=0)\n    torch.testing.assert_close(boxes, torch.tensor([[222., 7., 300., 142.]]), rtol=0, atol=0)\n    restored_image, restored_boxes = flip_image_target(reflected, boxes)\n    torch.testing.assert_close(restored_image, tensors[0], rtol=0, atol=0)\n    torch.testing.assert_close(restored_boxes, targets[0][\'boxes\'], rtol=0, atol=0)\n    # Optimizer updates match baseline SGD. Time schedule is the new shared rule.\n    parameter = torch.nn.Parameter(torch.ones(2))\n    reference_parameter = torch.nn.Parameter(torch.ones(2))\n    adapter = object.__new__(AgreedYoloTrainer)\n    tiny = torch.nn.ParameterList([parameter])\n    adapter.optimizer = adapter.build_optimizer(tiny)\n    reference_optimizer = torch.optim.SGD([reference_parameter], lr=0.005,\n        momentum=0.9, weight_decay=0.0005, nesterov=False)\n    for elapsed in [0, 3600, 5400, 9000, 10800]:\n        lr = budget_cosine_lr(elapsed, initial_lr=0.005, minimum_lr=1e-6)\n        for group in adapter.optimizer.param_groups:\n            group[\'lr\'] = lr\n        for group in reference_optimizer.param_groups:\n            group[\'lr\'] = lr\n        assert adapter.optimizer.param_groups[0][\'lr\'] == reference_optimizer.param_groups[0][\'lr\']\n        parameter.grad = torch.ones(2)\n        reference_parameter.grad = torch.ones(2)\n        adapter.optimizer.step()\n        reference_optimizer.step()\n        torch.testing.assert_close(parameter, reference_parameter, rtol=0, atol=0)\n        assert lr >= 1e-6\n    print(\'Input tensors, boxes, flip, SGD updates and budget schedule: OK\')\n\n', encoding='utf-8')
(OUT/'main_model.py').write_text('"""D-Fire main model experiments. Importable helpers; run through the companion notebook."""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nimport time\n\nNAMES = {0: \'smoke\', 1: \'fire\'}\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'}\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n\n\ndef write_csv(path, rows, fields=None):\n    rows = list(rows)\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields or list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef clean_label(text):\n    """Match Data Lead policy: reject invalid rows, clip valid boxes to [0, 1]."""\n    kept, rejected, clipped = [], [], 0\n    for number, line in enumerate(text.splitlines(), 1):\n        if not line.strip():\n            continue\n        try:\n            values = list(map(float, line.split()))\n            if len(values) != 5 or not all(map(math.isfinite, values)):\n                raise ValueError(\'expected five finite numbers\')\n            c, x, y, w, h = values\n            if c not in NAMES or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):\n                raise ValueError(\'invalid class/center/size\')\n            corners = [x-w/2, y-h/2, x+w/2, y+h/2]\n            a, b, d, e = [min(1., max(0., z)) for z in corners]\n            if d <= a or e <= b:\n                raise ValueError(\'no visible area\')\n            clipped += int(corners != [a, b, d, e])\n            kept.append(f\'{int(c)} {(a+d)/2:.8f} {(b+e)/2:.8f} {d-a:.8f} {e-b:.8f}\')\n        except ValueError as exc:\n            rejected.append({\'line\': number, \'text\': line, \'reason\': str(exc)})\n    return \'\\n\'.join(kept) + (\'\\n\' if kept else \'\'), rejected, clipped\n\n\ndef images(root, split):\n    return sorted(p for p in (Path(root)/split/\'images\').rglob(\'*\')\n                  if p.is_file() and p.suffix.lower() in EXTENSIONS)\n\n\ndef label_for(root, split, image):\n    relative = Path(image).relative_to(Path(root)/split/\'images\')\n    return Path(root)/split/\'labels\'/relative.with_suffix(\'.txt\')\n\n\ndef locate_prepared(downloaded):\n    """Find real image/label folders without trusting stale absolute paths in YAML."""\n    downloaded = Path(downloaded)\n    candidates = [downloaded] + [p.parent for p in downloaded.rglob(\'train\') if p.is_dir()]\n    candidates = sorted(set(p.resolve() for p in candidates if all(\n        (p/s/\'images\').is_dir() and (p/s/\'labels\').is_dir() for s in (\'train\', \'val\', \'test\'))))\n    if len(candidates) != 1:\n        raise RuntimeError(f\'Expected one prepared data root, found {candidates}. \'\n                           \'Set SOURCE_DATA to the folder containing train/, val/, test/. \'\n                           \'If the Input contains only an archive, extract it into working first.\')\n    return candidates[0]\n\n\ndef check_class_mapping(source, expected, search_root=None):\n    """Verify published names where available; otherwise record explicit project mapping."""\n    source = Path(source).resolve()\n    search_root = Path(search_root).resolve() if search_root is not None else source.parent\n    configs = sorted(set(search_root.rglob(\'data*.yaml\')) | set(search_root.rglob(\'data*.yml\')))\n    verified_files = []\n    for path in configs:\n        import yaml\n        cfg = yaml.safe_load(path.read_text(encoding=\'utf-8\'))\n        if not isinstance(cfg, dict) or \'names\' not in cfg:\n            continue\n        names = cfg[\'names\']\n        names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}\n        if names != expected:\n            raise ValueError(f\'Unexpected class mapping in {path}: {names}; expected {expected}\')\n        verified_files.append(str(path))\n    if not verified_files:\n        print(f\'No published YAML with names found. Using explicit project mapping: {expected}. \'\n              \'Label IDs will be validated; their semantic names cannot be inferred from numbers alone.\')\n    return dict(names=expected, verified_from_yaml=bool(verified_files), yaml_files=verified_files,\n                mapping_source=\'published_yaml\' if verified_files else \'explicit_project_configuration\')\n\n\ndef prepare(source, dest, expected=None, prepared=False):\n    """Writable image COPIES avoid Ultralytics repairing JPEGs in read-only inputs."""\n    from PIL import Image\n    source, dest = Path(source).resolve(), Path(dest).resolve()\n    if dest == source or source in dest.parents or dest in source.parents:\n        raise ValueError(\'Source and prepared output must be separate directory trees\')\n    if dest.exists():\n        raise FileExistsError(f\'{dest} already exists. Reuse its config or choose a new output.\')\n    dest.mkdir(parents=True)\n    report, rejects, manifest, hashes = {}, [], [], {}\n    for split in (\'train\', \'val\', \'test\'):\n        files = images(source, split)\n        if not files or (expected and len(files) != expected[split]):\n            raise ValueError(f\'{split}: unexpected image count {len(files)}\')\n        seen_labels, stats = set(), dict(images=len(files), boxes=0, empty_labels=0, clipped=0, rejected=0)\n        for src in files:\n            label = label_for(source, split, src)\n            if not label.is_file():\n                raise FileNotFoundError(f\'Missing label is not a verified negative: {label}\')\n            key = label.relative_to(source/split/\'labels\')\n            if key in seen_labels:\n                raise ValueError(f\'Ambiguous image-to-label mapping: {key}\')\n            seen_labels.add(key)\n            digest = hashlib.sha256(src.read_bytes()).hexdigest()\n            if digest in hashes and hashes[digest] != split:\n                raise ValueError(f\'Exact cross-split duplicate: {src}; other split={hashes[digest]}\')\n            hashes[digest] = split\n            rel = src.relative_to(source/split/\'images\')\n            target = dest/split/\'images\'/rel\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Verify decode without resizing or EXIF transpose (boxes use stored pixel coordinates).\n            with Image.open(src) as im:\n                im.load()\n            shutil.copy2(src, target)\n            original_text = label.read_text(encoding=\'utf-8\')\n            text, bad, clipped = clean_label(original_text)\n            if prepared:\n                original_rows = [list(map(float, line.split())) for line in original_text.splitlines() if line.strip()] if not bad else []\n                cleaned_rows = [list(map(float, line.split())) for line in text.splitlines()]\n                changed = len(original_rows) != len(cleaned_rows) or any(\n                    abs(a-b) > 1e-7 for row_a, row_b in zip(original_rows, cleaned_rows)\n                    for a, b in zip(row_a, row_b))\n                if bad or changed:\n                    raise ValueError(f\'Prepared label failed validation: {label}. \'\n                                     \'No automatic annotation changes are allowed in prepared mode.\')\n                text, clipped = original_text, 0\n            out_label = label_for(dest, split, target)\n            out_label.parent.mkdir(parents=True, exist_ok=True)\n            out_label.write_text(text, encoding=\'utf-8\')\n            stats[\'boxes\'] += sum(bool(line.strip()) for line in text.splitlines())\n            stats[\'empty_labels\'] += int(not text.strip())\n            stats[\'clipped\'] += clipped\n            stats[\'rejected\'] += len(bad)\n            rejects.extend(dict(split=split, image=str(rel), **row) for row in bad)\n            manifest.append(dict(split=split, image=str(rel), sha256=digest))\n        report[split] = stats\n        print(split, stats)\n    # Preserve preparation provenance where supplied by the published dataset.\n    for directory in (source.parent, source):\n        for name in (\'removed_label_rows.csv\', \'dataset_summary.json\', \'data_lead_final_summary.json\'):\n            report_path = directory/name\n            if report_path.is_file():\n                (dest/\'source_reports\').mkdir(exist_ok=True)\n                shutil.copy2(report_path, dest/\'source_reports\'/name)\n    write_csv(dest/\'rejected_rows.csv\', rejects, [\'split\', \'image\', \'line\', \'text\', \'reason\'])\n    write_csv(dest/\'manifest.csv\', manifest)\n    dump(dest/\'preparation.json\', dict(splits=report, prepared_input=prepared,\n         source=str(source), labels_modified=not prepared))  # Completion marker written last.\n    return report\n\n\ndef make_yaml(path, root, train=None):\n    import yaml\n    config = dict(path=str(Path(root).resolve()), train=train or \'train/images\',\n                  val=\'val/images\', test=\'test/images\', names=NAMES, nc=2)\n    Path(path).write_text(yaml.safe_dump(config, sort_keys=False), encoding=\'utf-8\')\n    return str(Path(path).resolve())\n\n\ndef load_model(weights, architecture=\'yolo\'):\n    from ultralytics import YOLO, RTDETR\n    return (RTDETR if architecture == \'rtdetr\' else YOLO)(str(weights))\n\n\ndef train(weights, data, out, name, epochs=50, batch=8, lr=0.001,\n          architecture=\'yolo\', patience=8, resume=None):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a Kaggle GPU accelerator before training.\')\n    run = Path(out)/name\n    if resume:\n        model = load_model(resume, architecture)\n        model.train(resume=True, device=0)\n    else:\n        if run.exists():\n            raise FileExistsError(f\'{run} exists. Use resume=last.pt or a new name.\')\n        model = load_model(weights, architecture)\n        start = time.time()\n        model.train(data=str(data), epochs=epochs, imgsz=640, batch=batch,\n                    device=0, workers=2, optimizer=\'AdamW\', lr0=lr, lrf=0.01,\n                    weight_decay=0.0005, warmup_epochs=3, cos_lr=True,\n                    seed=42, deterministic=True, amp=True, cache=False,\n                    patience=patience, close_mosaic=min(5, epochs),\n                    mosaic=1.0, mixup=0.0, fliplr=0.5, flipud=0.0,\n                    save=True, save_period=5, plots=True,\n                    project=str(out), name=name, exist_ok=False)\n        dump(run/\'elapsed.json\', {\'train_seconds\': time.time()-start})\n    save_dir = Path(model.trainer.save_dir)\n    best = save_dir/\'weights\'/\'best.pt\'\n    if not best.is_file():\n        raise FileNotFoundError(best)\n    del model\n    import gc\n    gc.collect()\n    torch.cuda.empty_cache()\n    return str(best)\n\n\ndef evaluate(weights, data, out, name, size=640, split=\'val\', architecture=\'yolo\'):\n    model = load_model(weights, architecture)\n    metrics = model.val(data=str(data), split=split, imgsz=size, batch=4,\n                        device=0, workers=2, conf=0.001, iou=0.7, max_det=300,\n                        rect=False, plots=True, project=str(out), name=name,\n                        exist_ok=False)\n    row = dict(name=name, checkpoint=str(weights), architecture=architecture,\n               split=split, imgsz=size, map50=float(metrics.box.map50),\n               map50_95=float(metrics.box.map),\n               inference_ms_per_image=float(metrics.speed[\'inference\']))\n    # AP class indices are explicit; absent classes must not inherit the mean AP.\n    for cls, ap in zip(metrics.box.ap_class_index, metrics.box.ap):\n        row[f\'{NAMES[int(cls)]}_map50_95\'] = float(ap)\n    dump(Path(metrics.save_dir)/\'summary.json\', row)\n    return row\n\n\ndef ground_truth(root, split, image):\n    result = []\n    for line in label_for(root, split, image).read_text().splitlines():\n        c, x, y, w, h = map(float, line.split())\n        result.append([int(c), x-w/2, y-h/2, x+w/2, y+h/2])\n    return result\n\n\ndef predict_records(weights, root, split, size=640, architecture=\'yolo\', subset=None):\n    model = load_model(weights, architecture)\n    paths = subset if subset is not None else images(root, split)\n    records = []\n    # Bound the source list per call; stream results, retaining only boxes, not image arrays.\n    for start in range(0, len(paths), 128):\n        results = model.predict(source=[str(p) for p in paths[start:start+128]],\n                                stream=True, imgsz=size, batch=4, device=0,\n                                conf=0.001, iou=0.7, max_det=300, rect=False, verbose=False)\n        for result in results:\n            boxes = result.boxes\n            predictions = [[int(c), float(s), *map(float, b)] for c, s, b in\n                           zip(boxes.cls.cpu().tolist(), boxes.conf.cpu().tolist(),\n                               boxes.xyxyn.cpu().tolist())]\n            records.append(dict(image=str(result.path),\n                                gt=ground_truth(root, split, result.path), pred=predictions))\n    if len(records) != len(paths):\n        raise RuntimeError(\'Prediction count mismatch\')\n    return records\n\n\ndef iou(a, b):\n    intersection = max(0., min(a[2], b[2])-max(a[0], b[0])) * max(0., min(a[3], b[3])-max(a[1], b[1]))\n    area = lambda box: max(0., box[2]-box[0])*max(0., box[3]-box[1])\n    union = area(a)+area(b)-intersection\n    return intersection/union if union else 0.\n\n\ndef operating_metrics(records, threshold, cls=None):\n    tp = fp = fn = negative_count = false_alarms = 0\n    for record in records:\n        gt = [g for g in record[\'gt\'] if cls is None or g[0] == cls]\n        predictions = sorted([p for p in record[\'pred\'] if p[1] >= threshold and\n                              (cls is None or p[0] == cls)], key=lambda p: -p[1])\n        used = set()\n        for p in predictions:\n            candidates = [(iou(p[2:], g[1:]), j) for j, g in enumerate(gt)\n                          if j not in used and g[0] == p[0]]\n            overlap, j = max(candidates, default=(0., -1))\n            if overlap >= 0.5:\n                tp += 1\n                used.add(j)\n            else:\n                fp += 1\n        fn += len(gt)-len(used)\n        # Negative means no annotations of either class, including for per-class FAR.\n        if not record[\'gt\']:\n            negative_count += 1\n            false_alarms += int(bool(predictions))\n    precision = tp/(tp+fp) if tp+fp else 0.\n    recall = tp/(tp+fn) if tp+fn else 0.\n    return dict(threshold=threshold, class_name=NAMES.get(cls, \'all\'), tp=tp, fp=fp, fn=fn,\n                precision=precision, recall=recall,\n                f1=2*precision*recall/(precision+recall) if precision+recall else 0.,\n                negative_images=negative_count, false_alarm_images=false_alarms,\n                false_alarm_rate=false_alarms/negative_count if negative_count else None)\n\n\ndef mine(weights, root, out, limit=200):\n    """Rank prepared empty-label train images; exclude known rejected-label cases from audit reports."""\n    source = Path(root)\n    negatives = [p for p in images(source, \'train\') if not label_for(source, \'train\', p).read_text().strip()]\n    # preparation\'s rejected_rows prevents mislabeled positives from becoming hard negatives.\n    rejected = set()\n    for report_path in (source/\'rejected_rows.csv\', source/\'source_reports\'/\'removed_label_rows.csv\'):\n        if report_path.is_file():\n            with report_path.open(encoding=\'utf-8\') as f:\n                rejected.update(r[\'image\'].replace(\'\\\\\', \'/\') for r in csv.DictReader(f)\n                                if r.get(\'split\') == \'train\' and r.get(\'image\'))\n    negatives = [p for p in negatives if p.relative_to(source/\'train\'/\'images\').as_posix() not in rejected]\n    records = predict_records(weights, root, \'train\', subset=negatives)\n    ranked = sorted(records, key=lambda r: max([p[1] for p in r[\'pred\']], default=0.), reverse=True)\n    rows = [dict(image=r[\'image\'], max_conf=max(p[1] for p in r[\'pred\']),\n                 approved=\'\', category=\'\', notes=\'\') for r in ranked if r[\'pred\']][:limit]\n    write_csv(Path(out)/\'hard_negative_review.csv\', rows,\n              [\'image\', \'max_conf\', \'approved\', \'category\', \'notes\'])\n    # Paginated contact sheets; inspect full images separately when uncertain.\n    import matplotlib.pyplot as plt\n    from PIL import Image\n    for start in range(0, len(rows), 20):\n        fig, axes = plt.subplots(4, 5, figsize=(18, 12))\n        for ax, row in zip(axes.flat, rows[start:start+20]):\n            with Image.open(row[\'image\']) as im:\n                ax.imshow(im.convert(\'RGB\'))\n            ax.set_title(f"{Path(row[\'image\']).name}\\nscore={row[\'max_conf\']:.3f}", fontsize=8)\n        for ax in axes.flat:\n            ax.axis(\'off\')\n        fig.tight_layout()\n        fig.savefig(Path(out)/f\'hard_negative_review_{start//20:02d}.jpg\', dpi=100)\n        plt.close(fig)\n    return rows\n\n\ndef ablation_lists(root, review, out, repeats=2):\n    """Equal length = equal optimizer steps; base data retained in both arms."""\n    base = [str(p.resolve()) for p in images(root, \'train\')]\n    membership = set(base)\n    with Path(review).open(encoding=\'utf-8\') as f:\n        reviewed = list(csv.DictReader(f))\n    approved = sorted({str(Path(r[\'image\']).resolve()) for r in reviewed\n                       if r[\'approved\'].strip().lower() == \'yes\'})\n    if not approved:\n        raise ValueError(\'Review candidates first; set approved=yes only for confirmed backgrounds.\')\n    for p in approved:\n        if p not in membership or label_for(root, \'train\', p).read_text().strip():\n            raise ValueError(f\'Approved image is not an empty-label TRAIN image: {p}\')\n    negatives = [p for p in base if not label_for(root, \'train\', p).read_text().strip()]\n    count = len(approved)*repeats\n    rng = random.Random(42)\n    # Controls repeat randomly sampled negatives; treatment repeats reviewed hard negatives.\n    control = base + rng.choices(negatives, k=count)\n    treatment = base + approved*repeats\n    assert len(control) == len(treatment)\n    paths = []\n    for name, entries in [(\'control\', control), (\'hard_negative\', treatment)]:\n        listing = Path(out)/f\'{name}_train.txt\'\n        listing.write_text(\'\\n\'.join(entries)+\'\\n\', encoding=\'utf-8\')\n        paths.append(make_yaml(Path(out)/f\'{name}.yaml\', root, str(listing.resolve())))\n    dump(Path(out)/\'ablation_design.json\', dict(base_images=len(base), extra_samples=count,\n         images_per_epoch=len(control), approved=approved, seed=42,\n         policy=\'same checkpoint, epochs, optimizer, batch; random negatives vs reviewed hard negatives\'))\n    return paths\n', encoding='utf-8')
(OUT/'final_evaluation.py').write_text('"""Common metrics for records with original-pixel xyxy boxes; no GPU imports at module level."""\nfrom pathlib import Path\nimport json\nfrom main_model import iou, operating_metrics\n\n\ndef matched_counts(record, threshold):\n    predictions = sorted((p for p in record[\'pred\'] if p[1] >= threshold), key=lambda p: -p[1])\n    used, matched = set(), set()\n    for k, p in enumerate(predictions):\n        overlap, j = max(((iou(p[2:], g[1:]), j) for j, g in enumerate(record[\'gt\'])\n                          if j not in used and g[0] == p[0]), default=(0.0, -1))\n        if overlap >= 0.5:\n            used.add(j)\n            matched.add(k)\n    return predictions, used, matched\n\n\ndef operating_rows(records, threshold):\n    rows = []\n    empty = [r for r in records if not r[\'gt\']]\n    for cls in (None, 0, 1):\n        row = operating_metrics(records, threshold, cls)\n        empty_fp = sum(sum(p[1] >= threshold and (cls is None or p[0] == cls)\n                           for p in r[\'pred\']) for r in empty)\n        row.update(empty_image_fp=empty_fp,\n                   fp_per_empty_image=empty_fp/len(empty) if empty else None)\n        rows.append(row)\n    return rows\n\n\ndef confusion_matrix(records, threshold):\n    # Rows=true, columns=predicted: smoke, fire, background.\n    # Preserve EXACTLY the class-aware TP matching used for P/R/F1 first.\n    matrix = [[0]*3 for _ in range(3)]\n    for record in records:\n        predictions, used, matched = matched_counts(record, threshold)\n        for k in matched:\n            cls = predictions[k][0]\n            matrix[cls][cls] += 1\n        for k, p in enumerate(predictions):\n            if k in matched:\n                continue\n            overlap, j = max(((iou(p[2:], g[1:]), j) for j, g in enumerate(record[\'gt\'])\n                              if j not in used and g[0] != p[0]), default=(0.0, -1))\n            if overlap >= 0.5:\n                used.add(j)\n                matrix[record[\'gt\'][j][0]][p[0]] += 1\n            else:\n                matrix[2][p[0]] += 1\n        for j, g in enumerate(record[\'gt\']):\n            if j not in used:\n                matrix[g[0]][2] += 1\n    return dict(labels=[\'smoke\', \'fire\', \'background\'], rows=\'true\', columns=\'predicted\',\n                background_background=\'not defined for object detection\', values=matrix)\n\n\ndef collect_records(predict_fn, root, split, batch_size, out, label):\n    import time\n    import torch\n    from PIL import Image\n    from main_model import images, ground_truth\n    paths = images(root, split)\n    with Image.open(paths[0]) as im:\n        warmup = [im.convert(\'RGB\') for _ in range(batch_size)]\n    for _ in range(3):\n        predict_fn(warmup)\n    torch.cuda.synchronize()\n    records, seconds = [], 0.0\n    with (Path(out)/f\'{label}_{split}_predictions.jsonl\').open(\'w\', encoding=\'utf-8\') as log:\n        for start in range(0, len(paths), batch_size):\n            part = paths[start:start+batch_size]\n            batch = []\n            for path in part:\n                with Image.open(path) as im:\n                    batch.append(im.convert(\'RGB\'))\n            torch.cuda.synchronize()\n            t0 = time.perf_counter()\n            with torch.inference_mode():\n                outputs = predict_fn(batch)\n            torch.cuda.synchronize()\n            seconds += time.perf_counter()-t0\n            for path, image, output in zip(part, batch, outputs):\n                w, h = image.size\n                truth = [[g[0], g[1]*w, g[2]*h, g[3]*w, g[4]*h]\n                         for g in ground_truth(root, split, path)]\n                record = dict(image=str(path), gt=truth, pred=[\n                    [int(c), float(s), *box] for c, s, box in zip(\n                        output[\'labels\'].detach().cpu().tolist(),\n                        output[\'scores\'].detach().cpu().tolist(),\n                        output[\'boxes\'].detach().cpu().tolist())])\n                records.append(record)\n                log.write(json.dumps(record)+\'\\n\')\n            if start % 200 == 0:\n                print(label, split, min(start+batch_size, len(paths)), \'/\', len(paths))\n    assert len(records) == len(paths)\n    return records, dict(batch=batch_size, warmup_batches=3, seconds=seconds,\n        ms_per_image=seconds/len(records)*1000, fps=len(records)/seconds,\n        scope=\'RGB PIL -> preprocessing, H2D, inference, postprocessing; excludes decode, D2H outputs, metrics\')\n\n\ndef coco_metrics(records):\n    import torch\n    from torchmetrics.detection.mean_ap import MeanAveragePrecision\n    metric = MeanAveragePrecision(box_format=\'xyxy\', iou_type=\'bbox\', class_metrics=True,\n                                  max_detection_thresholds=[1, 10, 100])\n    for r in records:\n        pred = dict(boxes=torch.tensor([p[2:] for p in r[\'pred\']], dtype=torch.float32).reshape(-1, 4),\n                    labels=torch.tensor([p[0] for p in r[\'pred\']], dtype=torch.long),\n                    scores=torch.tensor([p[1] for p in r[\'pred\']], dtype=torch.float32))\n        gt = dict(boxes=torch.tensor([g[1:] for g in r[\'gt\']], dtype=torch.float32).reshape(-1, 4),\n                  labels=torch.tensor([g[0] for g in r[\'gt\']], dtype=torch.long))\n        metric.update([pred], [gt])\n    result = metric.compute()\n    return dict(map50=float(result[\'map_50\']), map50_95=float(result[\'map\']),\n                per_class_map50_95={[\'smoke\',\'fire\'][int(c)]: float(v)\n                    for c, v in zip(result[\'classes\'].tolist(), result[\'map_per_class\'].tolist())})\n\n\ndef save_evaluation(records, threshold, timing, out, name):\n    import matplotlib.pyplot as plt\n    from main_model import dump, write_csv\n    out = Path(out)\n    operating = operating_rows(records, threshold)\n    matrix = confusion_matrix(records, threshold)\n    report = dict(metrics=coco_metrics(records), operating=operating,\n                  confusion_matrix=matrix, timing=timing, confidence=threshold,\n                  matching_iou=0.5, images=len(records),\n                  fp_per_empty_image_target_passed=(operating[0][\'fp_per_empty_image\'] < 0.5)\n                  if operating[0][\'fp_per_empty_image\'] is not None else None)\n    dump(out/f\'{name}_metrics.json\', report)\n    write_csv(out/f\'{name}_operating.csv\', operating)\n    fig, ax = plt.subplots(figsize=(5, 4))\n    ax.imshow(matrix[\'values\'], cmap=\'Blues\')\n    ax.set(xticks=range(3), yticks=range(3), xticklabels=matrix[\'labels\'],\n           yticklabels=matrix[\'labels\'], xlabel=\'Predicted\', ylabel=\'True\')\n    for i in range(3):\n        for j in range(3):\n            ax.text(j, i, str(matrix[\'values\'][i][j]) if (i,j)!=(2,2) else \'N/A\',\n                    ha=\'center\', va=\'center\')\n    fig.tight_layout()\n    fig.savefig(out/f\'{name}_confusion.png\', dpi=150)\n    plt.close(fig)\n    # Candidates for manual failure analysis; do not claim reviewed error categories.\n    candidates = []\n    for record in records:\n        predictions, used, matched = matched_counts(record, threshold)\n        fp, fn = len(predictions)-len(matched), len(record[\'gt\'])-len(used)\n        if fp or fn:\n            candidates.append((fp+fn, fp, fn, record))\n    candidates.sort(key=lambda item: -item[0])\n    from PIL import Image, ImageDraw\n    manifest = []\n    for index, (_, fp, fn, record) in enumerate(candidates[:6]):\n        with Image.open(record[\'image\']) as im:\n            image = im.convert(\'RGB\')\n        draw = ImageDraw.Draw(image)\n        for gt in record[\'gt\']:\n            draw.rectangle(gt[1:], outline=\'lime\', width=2)\n        for p in record[\'pred\']:\n            if p[1] >= threshold:\n                draw.rectangle(p[2:], outline=\'red\', width=2)\n                draw.text((p[2], p[3]), f\'{["smoke","fire"][p[0]]} {p[1]:.2f}\', fill=\'red\')\n        filename = f\'{name}_error_candidate_{index:02d}.jpg\'\n        image.save(out/filename)\n        manifest.append(dict(image=record[\'image\'], fp=fp, fn=fn, visualization=filename,\n                             review_notes=\'\'))\n    write_csv(out/f\'{name}_error_candidates.csv\', manifest,\n              [\'image\', \'fp\', \'fn\', \'visualization\', \'review_notes\'])\n    return report\n\n\ndef choose_threshold(records, out, name):\n    import matplotlib.pyplot as plt\n    from main_model import write_csv, dump\n    thresholds = [0.001, 0.01, 0.025] + [v/100 for v in range(5, 100, 5)]\n    rows = [dict(row) for threshold in thresholds for row in operating_rows(records, threshold)]\n    write_csv(Path(out)/f\'{name}_validation_threshold_sweep.csv\', rows)\n    overall = [r for r in rows if r[\'class_name\'] == \'all\']\n    best = max(overall, key=lambda r: (r[\'f1\'], r[\'recall\'], r[\'threshold\']))\n    dump(Path(out)/f\'{name}_locked_threshold.json\', dict(threshold=best[\'threshold\'],\n        split=\'val\', criterion=\'maximum micro-F1; ties: recall, then higher threshold\', matching_iou=0.5))\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    for cls in [\'all\', \'smoke\', \'fire\']:\n        part = [r for r in rows if r[\'class_name\'] == cls]\n        axes[0].plot([r[\'recall\'] for r in part], [r[\'precision\'] for r in part], label=cls)\n        axes[1].plot([r[\'threshold\'] for r in part], [r[\'f1\'] for r in part], label=cls)\n    axes[0].set(xlabel=\'Recall\', ylabel=\'Precision\', xlim=(0, 1), ylim=(0, 1))\n    axes[1].set(xlabel=\'Confidence threshold\', ylabel=\'F1\', ylim=(0, 1))\n    for ax in axes:\n        ax.legend()\n        ax.grid(alpha=0.2)\n    fig.tight_layout()\n    fig.savefig(Path(out)/f\'{name}_validation_sensitivity.png\', dpi=150)\n    plt.close(fig)\n    return best[\'threshold\']\n', encoding='utf-8')

import sys
sys.path.insert(0, str(OUT))

dump(OUT/'baseline_budget_reference.json', BASELINE_REFERENCE)
assert abs(HOURS * 3600 - BASELINE_SECONDS) < 1e-6
assert budget_cosine_lr(BASELINE_SECONDS, BASELINE_SECONDS) == 1e-6
print('Training+validation budget:', BASELINE_SECONDS, 'seconds; GPU:', torch.cuda.get_device_name(0))

(OUT/'RUN_PROTOCOL_EN.md').write_text('# YOLOv8s: retraining with the baseline time budget\n\nUse YOLOv8s_BASELINE_TIME_KAGGLE.ipynb for the new run.\nThe previous three-hour experiment and its results remain separate.\n\n## Changes\n\n- The baseline history records elapsed_sec=14488.2: 4 h 1 min 28.2 s for training\n  and validation. Reference: https://github.com/cvteam19/cv_project/blob/main/baseline/training_history.json\n- This YOLO run sets time=14488.2/3600 hours. Cosine LR decay spans that budget,\n  retaining initial LR 0.005 and minimum LR 0.000001.\n- Outputs use /kaggle/working/yolov8s_baseline_time and a separate artifact ZIP.\n- Start fresh from COCO-pretrained yolov8s.pt. Do not resume selected_best.pt:\n  the previous optimizer followed a different learning-rate schedule.\n- Use one Tesla T4. Equal elapsed time on different GPUs does not provide the\n  same computational budget.\n\nAll remaining settings are unchanged: SGD momentum=0.9, weight_decay=0.0005,\nNesterov=False, batch=4, FP32 without AMP or gradient accumulation, clipping=10,\njoint horizontal flips p=0.5, baseline input preprocessing, seed=42, no warmup\nand no EMA averaging. Select the checkpoint by validation COCO mAP@0.5:0.95.\n\nPatience remains 3. This matches the maximum available time: early stopping may\nfinish YOLO sooner. Report the actual duration from the new log rather than\npresenting the configured limit as the observed duration. The baseline completed\nsix epochs without triggering early stopping.\n\nThe final in-loop validation and concluding best-checkpoint evaluation may slightly\nexceed the limit. Dataset preparation and separate evaluation are additional.\nMatching the time budget does not align every condition: the baseline retains\ncosine decay over six epochs; YOLO retains time-based cosine decay.\n\n## Run and collect outputs\n\n1. Upload the notebook to Kaggle. Enable Internet and a Tesla T4; attach\n   linxel/dfire-prepared.\n2. Run the cells in order. Allow more than four hours of GPU quota for the\n   training loop plus concluding checks.\n3. Keep RUN_FINAL_TEST=False. Test results are not needed to assess validation\n   changes in this rerun.\n4. Save a Kaggle Version with outputs and yolov8s_baseline_time_artifacts.zip.\n5. Inspect the new archive for actual duration, selected epoch, validation mAP\n   and confidence threshold. Do not claim updated results or improvement before\n   the run completes.\n\nProvide the evaluation lead with the new checkpoint and final_yolo_protocol.py\nfor the common final evaluation. Preserve the old checkpoint as experiment\nhistory. Do not replace published results with unverified numbers.\n\nStatus: notebook prepared; syntax and configuration checks can run locally.\nThe new GPU training run has not been executed. Additional time does not\nguarantee better detection quality.\n', encoding='utf-8')

In [ ]:
from ultralytics import YOLO
TRAIN_ARGS = dict(
    data=str(DATA_YAML), project=str(OUT), name='train', exist_ok=False,
    epochs=1000, time=HOURS, patience=3,
    imgsz=640, batch=BATCH, nbs=BATCH, amp=False, device=0, workers=2,
    optimizer='SGD', lr0=0.005, lrf=0.0002, momentum=0.9,
    weight_decay=0.0005, warmup_epochs=0.0, warmup_bias_lr=0.0, cos_lr=False,
    seed=42, deterministic=True, cache=False, rect=False, multi_scale=False,
    mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, close_mosaic=0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, bgr=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    val=True, plots=True, save=True, save_period=-1,
    conf=0.001, iou=0.7, max_det=100,
)
dump(OUT/'requested_training_config.json', TRAIN_ARGS)
print(TRAIN_ARGS)

def verify_runtime(trainer):
    assert trainer.amp is False
    assert trainer.accumulate == 1
    assert trainer.batch_size == BATCH
    assert isinstance(trainer.train_loader.dataset, BaselineInputDataset)
    assert isinstance(trainer.validator, BaselineInputValidator)
    assert type(trainer.optimizer).__name__ == 'SGD'
    assert len(trainer.optimizer.param_groups) == 1
    group = trainer.optimizer.param_groups[0]
    assert group['momentum'] == 0.9 and group['weight_decay'] == 0.0005
    assert not group['nesterov']
    assert trainer.test_loader.batch_size == BATCH
    dump(OUT/'verified_runtime.json', dict(amp=False, batch=BATCH, accumulate=1,
        optimizer='SGD', nesterov=False, weight_decay=group['weight_decay'],
        warmup=False, ema=False, input='torchvision GeneralizedRCNNTransform',
        normalization=dict(mean=BASELINE_MEAN, std=BASELINE_STD),
        scheduler='cosine by elapsed training+validation time', budget_hours=HOURS))

assert not (OUT/'train').exists(), 'This run already exists. Preserve it and choose a different OUT for a new run.'
model = YOLO('yolov8s.pt')
model.add_callback('on_train_start', verify_runtime)
import time
started = time.perf_counter()
model.train(trainer=AgreedYoloTrainer, **TRAIN_ARGS)
elapsed = time.perf_counter()-started
BEST = Path(model.trainer.best)
assert BEST.is_file()
history = __import__('pandas').read_csv(model.trainer.csv)
history.columns = history.columns.str.strip()
dump(OUT/'training_summary.json', dict(
    elapsed_train_call_seconds=elapsed, logged_epochs=len(history),
    planned_time_limit_seconds=BASELINE_SECONDS, baseline_reference=BASELINE_REFERENCE,
    last_epoch_may_be_partial=True, best=str(BEST),
    best_fitness=float(model.trainer.best_fitness),
    selection='maximum validation torchmetrics COCO mAP@0.5:0.95',
    effective_training_args=json.loads(json.dumps(vars(model.trainer.args), default=str))))
print('Best checkpoint:', BEST)
display(history.tail())

## Validation: confidence, common metrics and error candidates
Select confidence separately for each model by maximizing validation micro-F1.
Export mAP50, mAP50:95, precision/recall/F1, TP/FP/FN for smoke, fire and micro totals,
FP per completely empty image, the confusion matrix, latency and throughput.
Check the FP/image target of less than 0.5: detected boxes on empty images divided
by the number of empty images. Timing uses batch 4 and excludes file decoding.
A single timing pass is preliminary and does not characterize a latency distribution.

The confusion matrix first preserves the class-aware matching used for P/R/F1,
then matches remaining overlapping boxes of different classes as classification errors.
Background/background is undefined. Save up to six error candidates: green boxes are
ground truth, red boxes are predictions. Manually inspect at least three representative
cases for the report; automatic candidate selection does not replace error analysis.

If BASELINE_WEIGHTS is provided, evaluate the corrected baseline with the same code
and select its own validation threshold. Supply the corrected checkpoint, not the old run.

In [ ]:
import shutil, gc
import pandas as pd
from final_evaluation import collect_records, choose_threshold, save_evaluation
selected = OUT/'selected_best.pt'
shutil.copy2(BEST, selected)

def load_yolo_for_evaluation():
    core = YOLO(str(selected)).model.to('cuda').float().eval()
    return core, lambda batch: predict_with_baseline_input(core, batch, device='cuda')

def load_baseline_for_evaluation():
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.transforms.functional import to_tensor
    network = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
        box_score_thresh=0.001, box_nms_thresh=0.7, box_detections_per_img=100)
    features = network.roi_heads.box_predictor.cls_score.in_features
    network.roi_heads.box_predictor = FastRCNNPredictor(features, 3)
    checkpoint = torch.load(BASELINE_WEIGHTS, map_location='cpu', weights_only=True)
    network.load_state_dict(checkpoint['model_state_dict'])
    network.transform.min_size = (640,)
    network.transform.max_size = 640
    network = network.to('cuda').float().eval()
    def predict(batch):
        outputs = network([to_tensor(im).to('cuda') for im in batch])
        return [dict(boxes=o['boxes'], scores=o['scores'], labels=o['labels']-1) for o in outputs]
    return network, predict

loaders = {'yolov8s': load_yolo_for_evaluation}
if BASELINE_WEIGHTS is not None:
    assert Path(BASELINE_WEIGHTS).is_file()
    loaders['faster_rcnn'] = load_baseline_for_evaluation

# Release the training model before loading either evaluation model.
del model
gc.collect()
torch.cuda.empty_cache()
locked, val_reports = {}, {}
for label, loader in loaders.items():
    network, predict_fn = loader()
    records, timing = collect_records(predict_fn, ROOT, 'val', BATCH, OUT, label)
    threshold = choose_threshold(records, OUT, label)
    locked[label] = dict(threshold=threshold, split='val', imgsz=640,
                         criterion='maximum validation micro-F1', batch=BATCH)
    val_reports[label] = save_evaluation(records, threshold, timing, OUT, label+'_val')
    display(pd.DataFrame(val_reports[label]['operating']))
    print(label, val_reports[label]['metrics'], timing)
    del predict_fn, network, records
    gc.collect()
    torch.cuda.empty_cache()
dump(OUT/'locked_selection.json', locked)

## Final test: only after model and settings are fixed
RUN_FINAL_TEST=False by default. Once model and setting selection is complete,
enable it in the configuration or in a separate cell immediately before this block.
Test evaluation uses only the thresholds saved in locked_selection.json; nothing
is selected on test. Do not change the recipe after viewing test results and then
reuse that test as an independent final evaluation. Evaluation without retraining
can be performed using the saved helper files and selected_best.pt from the ZIP.

In [ ]:
if RUN_FINAL_TEST:
    locked = json.loads((OUT/'locked_selection.json').read_text())
    reports = {}
    for label, loader in loaders.items():
        network, predict_fn = loader()
        records, timing = collect_records(predict_fn, ROOT, 'test', BATCH, OUT, label)
        reports[label] = save_evaluation(records, locked[label]['threshold'], timing,
                                         OUT, label+'_test')
        common_rows = __import__('final_evaluation').operating_rows(records, 0.5)
        write_csv(OUT/f'{label}_test_common_threshold_050.csv', common_rows)
        display(pd.DataFrame(reports[label]['operating']))
        print(label, reports[label]['metrics'], timing)
        del predict_fn, network, records
        gc.collect()
        torch.cuda.empty_cache()
    dump(OUT/'test_comparison.json', reports)
else:
    print('Test skipped. Validation outputs and locked thresholds are saved.')

In [ ]:
import zipfile
archive = Path('/kaggle/working/yolov8s_baseline_time_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob('*'):
        if path.is_file() and (path.suffix != '.pt' or path == selected):
            z.write(path, path.relative_to(OUT))
print('Save a Kaggle Version with outputs. Artifact archive:', archive)